# Simple CTE Display Test

最简单的测试：显示原始图像、CTE检测结果和mask


In [ ]:
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 添加父目录到路径
sys.path.insert(0, str(Path.cwd().parent))

from real_car_env import RealJetRacerEnv
from cte_estimator import VisualCTEEstimator

%matplotlib inline
plt.rcParams['figure.figsize'] = [15, 5]

print("✅ 导入完成")


In [ ]:
# 创建CTE估计器
cte_estimator = VisualCTEEstimator(
    method="centerline_tracking",  # 可选: "canny_edges", "color_edge_detection", "centerline_tracking"
    image_width=320,
    image_height=240,
    max_cte=3.0,
    track_lower=(10, 100, 100),  # HSV lower bound (用于 color_edge_detection 和 centerline_tracking)
    track_upper=(25, 255, 255),  # HSV upper bound (用于 color_edge_detection 和 centerline_tracking)
)

print("✅ CTE估计器创建完成")


In [ ]:
# 读取测试图片
test_image_path = Path('../real_road_data/pic0.jpg')

if test_image_path.exists():
    frame_bgr = cv2.imread(str(test_image_path))
    print(f"✅ 读取图片: {test_image_path}")
    print(f"   图片尺寸: {frame_bgr.shape}")
    print(f"   图片数据类型: {frame_bgr.dtype}")
    print(f"   图片值范围: [{frame_bgr.min()}, {frame_bgr.max()}]")
else:
    # 如果没有图片，创建一个测试图像
    frame_bgr = np.zeros((240, 320, 3), dtype=np.uint8)
    # 添加一些彩色区域用于测试
    frame_bgr[100:140, 100:220] = [0, 255, 255]  # 黄色区域（BGR格式）
    print("⚠️  未找到测试图片，使用生成的测试图像")
    print(f"   图片尺寸: {frame_bgr.shape}")


In [ ]:
# 执行CTE估计
cte, confidence = cte_estimator.estimate(frame_bgr)

print(f"CTE: {cte:.3f}")
print(f"Confidence: {confidence:.3f}")
print(f"Debug image available: {cte_estimator.last_debug_image is not None}")
print(f"Mask image available: {cte_estimator.last_mask_image is not None}")


In [ ]:
# 显示所有图像
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. 原始图像
print("显示原始图像...")
print(f"  frame_bgr shape: {frame_bgr.shape}, dtype: {frame_bgr.dtype}")
print(f"  frame_bgr min/max: {frame_bgr.min()}/{frame_bgr.max()}")

frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
print(f"  frame_rgb shape: {frame_rgb.shape}, dtype: {frame_rgb.dtype}")
print(f"  frame_rgb min/max: {frame_rgb.min()}/{frame_rgb.max()}")

axes[0].imshow(frame_rgb)
axes[0].set_title(f"Original Frame\nCTE: {cte:.2f}, Conf: {confidence:.2f}")
axes[0].axis('off')

# 2. CTE检测结果（debug image）
print("\n显示CTE检测结果...")
if cte_estimator.last_debug_image is not None:
    print(f"  debug_image shape: {cte_estimator.last_debug_image.shape}, dtype: {cte_estimator.last_debug_image.dtype}")
    debug_rgb = cv2.cvtColor(cte_estimator.last_debug_image, cv2.COLOR_BGR2RGB)
    axes[1].imshow(debug_rgb)
    axes[1].set_title("CTE Detection (Debug)")
else:
    axes[1].text(0.5, 0.5, 'No debug image', ha='center', va='center', fontsize=14)
    axes[1].set_title("CTE Detection (No data)")
axes[1].axis('off')

# 3. Mask图像
print("\n显示Mask图像...")
if cte_estimator.last_mask_image is not None:
    print(f"  mask_image shape: {cte_estimator.last_mask_image.shape}, dtype: {cte_estimator.last_mask_image.dtype}")
    print(f"  mask_image min/max: {cte_estimator.last_mask_image.min()}/{cte_estimator.last_mask_image.max()}")
    print(f"  mask非零像素数: {np.count_nonzero(cte_estimator.last_mask_image)}")
    axes[2].imshow(cte_estimator.last_mask_image, cmap='gray')
    axes[2].set_title("Mask (CTE Estimation)")
else:
    axes[2].text(0.5, 0.5, 'No mask image', ha='center', va='center', fontsize=14)
    axes[2].set_title("Mask (No data)")
axes[2].axis('off')

plt.tight_layout()
plt.show()

print("\n✅ 显示完成！")


## 测试 canny_edges 方法


In [ ]:
# 测试 canny_edges 方法
cte_estimator_canny = VisualCTEEstimator(
    method="canny_edges",
    image_width=320,
    image_height=240,
    max_cte=3.0,
)

cte_canny, confidence_canny = cte_estimator_canny.estimate(frame_bgr)

print(f"Canny Edges - CTE: {cte_canny:.3f}, Confidence: {confidence_canny:.3f}")

# 显示结果
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 原始图像
frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
axes[0].imshow(frame_rgb)
axes[0].set_title(f"Original Frame\nCTE: {cte_canny:.2f}, Conf: {confidence_canny:.2f}")
axes[0].axis('off')

# Debug图像
if cte_estimator_canny.last_debug_image is not None:
    debug_rgb = cv2.cvtColor(cte_estimator_canny.last_debug_image, cv2.COLOR_BGR2RGB)
    axes[1].imshow(debug_rgb)
    axes[1].set_title("CTE Detection (Debug)")
else:
    axes[1].text(0.5, 0.5, 'No debug image', ha='center', va='center')
    axes[1].set_title("CTE Detection (No data)")
axes[1].axis('off')

# Mask图像（Canny边缘检测结果）
if cte_estimator_canny.last_mask_image is not None:
    axes[2].imshow(cte_estimator_canny.last_mask_image, cmap='gray')
    axes[2].set_title("Mask (Canny Edges)")
else:
    axes[2].text(0.5, 0.5, 'No mask image', ha='center', va='center')
    axes[2].set_title("Mask (No data)")
axes[2].axis('off')

plt.tight_layout()
plt.show()
